In [1]:
! ls echojepa_cls/

ucmc22k-classifier-echojepa-224px-multi.csv
ucmc22k-classifier-echojepa-224px.csv
ucmc22k-classifier-echoprime-224px.csv
ucmc22k-classifier-panecho-224px.csv
ucmc22k-classifier-videomae-224px.csv


In [2]:
! cat echojepa_cls/ucmc22k-classifier-echojepa-224px-multi.csv | head -n 5

video_path,true_label,predicted_class,prediction_confidence
video_0_0,4.0,0,0.13574219
video_0_1,3.0,0,0.15722656
video_0_2,4.0,0,0.13867188
video_0_3,1.0,0,0.14453125
cat: write error: Broken pipe


In [3]:
! cat echojepa_cls/ucmc22k-classifier-echojepa-224px.csv | head -n 5

video_path,true_label,predicted_class,prediction_confidence
video_0_0,4.0,8,0.107421875
video_0_1,3.0,0,0.12060547
video_0_2,4.0,8,0.11376953
video_0_3,1.0,8,0.11279297
cat: write error: Broken pipe


In [5]:
import pandas as pd
import glob
import os

file_pattern = "echojepa_cls/*.csv"

print(f"{'File Name':<50} | {'Accuracy':<10}")
print("-" * 65)

for filepath in sorted(glob.glob(file_pattern)):
    try:
        df = pd.read_csv(filepath)
        
        # Drop any empty rows just in case
        df = df.dropna(subset=['true_label', 'predicted_class'])

        # FORCE CONVERSION:
        # 1. .astype(float) handles strings like "4.0" or numbers like 4.0
        # 2. .astype(int) strips the decimal (4.0 -> 4) so it matches the integers
        y_true = df['true_label'].astype(float).astype(int)
        y_pred = df['predicted_class'].astype(float).astype(int)
        
        # Calculate accuracy on the cleaned integer data
        accuracy = (y_true == y_pred).mean()
        
        filename = os.path.basename(filepath)
        print(f"{filename:<50} | {accuracy:.2%}")
        
    except Exception as e:
        print(f"{os.path.basename(filepath):<50} | Error: {e}")

File Name                                          | Accuracy  
-----------------------------------------------------------------
ucmc22k-classifier-echojepa-224px-multi.csv        | 6.20%
ucmc22k-classifier-echojepa-224px.csv              | 6.63%
ucmc22k-classifier-echoprime-224px.csv             | 9.74%
ucmc22k-classifier-panecho-224px.csv               | 2.02%
ucmc22k-classifier-videomae-224px.csv              | 5.22%


In [6]:
import pandas as pd
import glob
import os

file_pattern = "echojepa_cls/*.csv"

# Iterate through each file
for filepath in sorted(glob.glob(file_pattern)):
    print(f"\n{'='*60}")
    print(f"FILE: {os.path.basename(filepath)}")
    print(f"{'='*60}")
    
    try:
        df = pd.read_csv(filepath)
        
        # 1. Clean the data (Handle the float vs int issue)
        df = df.dropna(subset=['true_label', 'predicted_class'])
        df['true_label'] = df['true_label'].astype(float).astype(int)
        df['predicted_class'] = df['predicted_class'].astype(float).astype(int)
        
        # 2. Mark which rows are correct
        df['is_correct'] = df['true_label'] == df['predicted_class']
        
        # 3. Group by the True Label to get per-class stats
        # 'count' tells you how many samples were in that class
        # 'mean' calculates the percentage correct (Accuracy/Recall)
        class_analysis = df.groupby('true_label')['is_correct'].agg(['count', 'mean'])
        
        # 4. Rename and Format columns for readability
        class_analysis.columns = ['Sample Count', 'Class Accuracy']
        class_analysis['Class Accuracy'] = class_analysis['Class Accuracy'].map('{:.2%}'.format)
        
        print(class_analysis)
        
    except Exception as e:
        print(f"Could not process {os.path.basename(filepath)}: {e}")


FILE: ucmc22k-classifier-echojepa-224px-multi.csv
            Sample Count Class Accuracy
true_label                             
0                    255        100.00%
1                    519          0.00%
2                    335          0.00%
3                    305          0.00%
4                    618          0.00%
5                     83          0.00%
6                     23          0.00%
7                    341          0.00%
8                    300          0.00%
9                    442          0.00%
11                   878          0.00%
12                    17          0.00%

FILE: ucmc22k-classifier-echojepa-224px.csv
            Sample Count Class Accuracy
true_label                             
0                    255         45.49%
1                    519          0.00%
2                    335          0.00%
3                    305          0.00%
4                    618          0.32%
5                     83          8.43%
6                     23